# EXP16: Walk-Forward 滚动回测

用 2 年 In-Sample 训练 + 1 年 Out-of-Sample 测试，滚动推进。
验证策略是否在未见数据上持续盈利。

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Walk-Forward Windows
| Window | IS (2 yr) | OOS (1 yr) |
|---|---|---|
| WF1 | 2015-2016 | 2017 |
| WF2 | 2016-2017 | 2018 |
| WF3 | 2017-2018 | 2019 |
| WF4 | 2018-2019 | 2020 |
| WF5 | 2019-2020 | 2021 |
| WF6 | 2020-2021 | 2022 |
| WF7 | 2021-2022 | 2023 |
| WF8 | 2022-2023 | 2024 |
| WF9 | 2023-2024 | 2025-2026 |

In [ ]:
windows = [
    ("WF1", "2015-01-01", "2017-01-01", "2017-01-01", "2018-01-01"),
    ("WF2", "2016-01-01", "2018-01-01", "2018-01-01", "2019-01-01"),
    ("WF3", "2017-01-01", "2019-01-01", "2019-01-01", "2020-01-01"),
    ("WF4", "2018-01-01", "2020-01-01", "2020-01-01", "2021-01-01"),
    ("WF5", "2019-01-01", "2021-01-01", "2021-01-01", "2022-01-01"),
    ("WF6", "2020-01-01", "2022-01-01", "2022-01-01", "2023-01-01"),
    ("WF7", "2021-01-01", "2023-01-01", "2023-01-01", "2024-01-01"),
    ("WF8", "2022-01-01", "2024-01-01", "2024-01-01", "2025-01-01"),
    ("WF9", "2023-01-01", "2025-01-01", "2025-01-01", "2026-04-01"),
]

wf_results = []
for name, is_start, is_end, oos_start, oos_end in windows:
    is_data = data.slice(is_start, is_end)
    oos_data = data.slice(oos_start, oos_end)
    
    if len(is_data.m15_df) < 500 or len(oos_data.m15_df) < 500:
        print(f"{name}: skipped (insufficient data)")
        continue
    
    is_r = run_variant(is_data, f"{name}_IS", **LIVE_KWARGS)
    oos_r = run_variant(oos_data, f"{name}_OOS", **LIVE_KWARGS)
    
    wf_results.append({
        'window': name,
        'is_sharpe': is_r['sharpe'], 'is_pnl': is_r['total_pnl'], 'is_n': is_r['n'],
        'oos_sharpe': oos_r['sharpe'], 'oos_pnl': oos_r['total_pnl'], 'oos_n': oos_r['n'],
        'degradation': oos_r['sharpe'] - is_r['sharpe'],
    })
    print(f"{name}: IS Sharpe={is_r['sharpe']:.2f} ({is_r['n']} trades), "
          f"OOS Sharpe={oos_r['sharpe']:.2f} ({oos_r['n']} trades), "
          f"Deg={oos_r['sharpe']-is_r['sharpe']:+.2f}")

In [ ]:
wf_df = pd.DataFrame(wf_results)
print("\n=== Walk-Forward Summary ===")
print(wf_df.to_string(index=False))

oos_positive = (wf_df['oos_pnl'] > 0).sum()
print(f"\nOOS profitable: {oos_positive}/{len(wf_df)} windows")
print(f"Avg OOS Sharpe: {wf_df['oos_sharpe'].mean():.2f}")
print(f"Avg IS→OOS degradation: {wf_df['degradation'].mean():.2f}")
print(f"Combined OOS PnL: ${wf_df['oos_pnl'].sum():.0f}")